In [1]:
import numpy as np
from tqdm import tqdm

In [2]:
float_formatter = "{:.2f}".format
np.set_printoptions(formatter={'float_kind':float_formatter})

Important notes:

- N is per-action
- For the running average, N starts at 1, such that the initial value is fully replaced

In [30]:

def simulate(epsilon = 0.1, alpha = 0.01, steps = 1000, runs = 2000, alpha_decay = False):
    avg_reward = np.zeros((steps)) # average reward for each step
    optimal_action_count = np.zeros((steps)) # percentage of optimal action taken

    n = 1
    for run in tqdm(range(runs)):
        q = np.random.normal(0, 1, (10))
        Q = np.zeros((10))
        N = np.ones((10))

        for i in range(steps):
            q = q + np.random.normal(0, 0.01, (10)) # random walk
            optimal_action = np.argmax(q)

            if np.random.random() >= epsilon: # take greedy action
                maxQ = Q.max()
                index = np.random.choice(np.flatnonzero(Q == maxQ))
            else: # explore
                index = np.random.randint(0,10)

            if index == optimal_action:
                optimal_action_count[i] += 1
            
            reward = np.random.normal(q[index], 1)

            if alpha_decay:
                Q[index] = Q[index] + (0.1) * (reward - Q[index])
            else:
                Q[index] = Q[index] + (1/N[index]) * (reward - Q[index])
            N[index] = N[index] + 1



            avg_reward[i] = avg_reward[i] + (1/n) * (reward - avg_reward[i])
        n += 1

    optimal_action_count /= runs
    
    return avg_reward, optimal_action_count

In [31]:
import plotly.express as px

data = {}
data['0.1'], data['0.1_opt'] = simulate(epsilon=0.1, steps=10000)
# data['0.01'], data['0.01_opt'] = simulate(epsilon=0.01, steps=10000)
# data['0'], data['0_opt'] = simulate(epsilon=0.0, steps=10000)
data['x'] = np.arange(1, 10001)


100%|██████████| 2000/2000 [03:05<00:00, 10.76it/s]


In [32]:
data['0.1_alpha'], data['0.1_opt_alpha'] = simulate(epsilon=0.1, steps=10000, alpha_decay=True)
# data['0.01_alpha'], data['0.01_opt_alpha'] = simulate(epsilon=0.01, steps=10000, alpha_decay=True)
# data['0_alpha'], data['0_opt_alpha'] = simulate(epsilon=0.0, steps=10000, alpha_decay=True)

100%|██████████| 2000/2000 [03:06<00:00, 10.71it/s]


In [35]:
# px.line(data, x='x', y=['0.1', '0.01', '0'])
# px.line(data, x='x', y=['0.1_opt', '0.01_opt', '0_opt'])
# px.line(data, x='x', y=['0.1', '0.1_alpha'])
px.line(data, x='x', y=['0.1_opt', '0.1_opt_alpha'])